# **1. Load and Explore Dataset**

In [135]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import zipfile
import warnings
warnings.filterwarnings('ignore')
import os

In [136]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [137]:
#extract zip
zip_path = '/content/drive/MyDrive/Fintech & BFSI.zip'
extract_path = '/content/Fintech'

if not os.path.exists(extract_path):
  with zipfile.ZipFile(zip_path, 'r') as  zip_ref:
    zip_ref.extractall(extract_path)

In [138]:
os.listdir('/content/Fintech')

['track1_merchants_master.csv',
 'track1_upi_transactions.csv',
 'track1_dataset_notes.txt',
 'track1_kyc_records.csv',
 'track1_chargebacks.json']

In [139]:
import json # as json file included in directory
import re  # regex for replace, extrach and serach [0-9 a-z A-Z] and special characters like (+ * @) and many more

In [140]:
transaction = pd.read_csv('Fintech/track1_upi_transactions.csv')
merchants = pd.read_csv('Fintech/track1_merchants_master.csv')
kyc = pd.read_csv('Fintech/track1_kyc_records.csv')

with open('Fintech/track1_chargebacks.json') as j:   # json -> tabular dataframe
  chargebacks = pd.json_normalize(json.load(j))

### Quick track of all records from each files

In [141]:
# quick check of all files
def showdata(df, filename):
  print(f"------{filename}------")
  print("\nShape: ", df.shape)
  print("Duplicated Rows: ", df.duplicated().sum())
  print("\nNull Values in each Column")
  print(df.isnull().sum())
  print()

showdata(transaction, "TRANSACTIONS")
showdata(kyc, "KYC RECORDS")
showdata(merchants, "MERCHANTS")
showdata(chargebacks, "CHARGEBACKS")


------TRANSACTIONS------

Shape:  (20400, 8)
Duplicated Rows:  400

Null Values in each Column
txn_id            0
timestamp         0
user_id           0
merchant_id       0
amount            0
utr            1024
mcc            2926
status            0
dtype: int64

------KYC RECORDS------

Shape:  (36400, 12)
Duplicated Rows:  278

Null Values in each Column
user_id                0
full_name              0
pan                 1896
aadhaar             2664
date_of_birth       2944
city                   0
state                  0
monthly_income      2933
occupation             0
signup_timestamp    2910
kyc_status             0
risk_segment           0
dtype: int64

------MERCHANTS------

Shape:  (6210, 11)
Duplicated Rows:  12

Null Values in each Column
merchant_id                    0
merchant_name                  0
mcc                          514
merchant_category              0
business_type                  0
city                           0
state                          0


# **EDA & Data Preprocessing**





**The list of Data Cleaning process we applied in the model is,**

1. **Standardization** - *Bringing data values into a consistent format so that same entity is represented in uniform way*
2.  **Normalization** - *converting data that is represented in different fromat into a common analytical fromat*
3.  **Paring** - *taking data stored in different or unstructured representations and converting it into a structured format*
4.  **Data Integrity** - *ensuring that the data remains accurate, consistent, valid and reliable throughout the cleaning and analysis process*
5. **Handling missing & Invalid values** - *values were handled appropriately during parsing and numeric conversion rather than simply deleting the entire dataset* ( **Nan, NaT** )

**Why we are not merging all the files for handling data, because**
*  Handling different data entities - files are csv and json fromat
*  Preventing Garbage in and Garbage Out data value's - prevents duplicated values
* Readability - can trace the changes before and after the EDA analysis
*   Memory and Computational Efficiency - saves time and takes less space

# 2. **Transaction File**


In [142]:
txn = transaction.copy() # for tracing the changes
txn.head()

,txn_id,timestamp,user_id,merchant_id,amount,utr,mcc,status
0,TXN00011869,2026-01-15 00:11:30,USR45826,MCH7045,15722.34,UTR6498104698,5411.0,COMPLETED
1,TXN00010383,2026-01-17 20:09:44,USR79397,MCH5031,Rs. 6362.9,UTR7656190355,4131.0,TXN_FAILED
2,TXN00008297,2026-01-23 14:10:16,USR87810,MCH9809,15446.19,UTR9037001889,5411.0,S
3,TXN00006448,1770063471,USR54287,MCH6928,12110.49,UTR5257823698,5411.0,S
4,TXN00018792,2026-03-31 14:02:37,USR53865,MCH8121,Rs. 19432.94,UTR4204272894,4131.0,TXN_SUCCESS


### As per my view from the data, the columns which required changes are,


1.    **txn_id** - for every transaction there is unique id or key will be generatedand [ *txn_id should be unique & same format TXN + integer format* ]


2.   **usr_id** and **merchant_id** - it can same as, same user and merchant can go through various payments [ *same format USR, MCH + integer fromat* ]
3.   **timestamp** - as follow columns follows human redable format [ *YYYY-MM-DD HH:MM:SS* ] and remaining are in Epoch Time [ *10 digit integer designed for systems and database* ]
4.   **amount** - inconsistent symbols like with [ *Rs. , Rs.   , and other symbols* ]
5.   **status** - need to be mapped to its belonging





### **2.1 txn_id column**

In [143]:
print("Total rows loaded in transaction file: ", len(txn))

Total rows loaded in transaction file:  20400


In [144]:
# just maintaining the first occurence of the txn_id, because every transaction id is unique
txn = txn.drop_duplicates(subset='txn_id', keep='first')
txn.shape

(20000, 8)



*   We have checked the uniquenss of the id [ *as kept only the first occurencre, remaining has been trimmed* ]
*   From it, we have removed **400 rows**



### **2.2 user_id, merchant_id and txn_id**

In [145]:
# for txn_id - we maintained title should be capital
txn['txn_id'] = (txn['txn_id'].astype('string').str.strip().str.upper())

# removed the trailing spaces, title to capital for user_id and merchant_id
txn['user_id'] = (txn['user_id'].astype('string').str.strip().str.upper()
       .str.replace(r'\s+', '', regex=True))

txn['merchant_id'] = (
    txn['merchant_id'].astype('string').str.strip()
    .str.upper().str.replace(r'\s+', '', regex=True))

### **2.3 timestamp**

As per timestamp for data, it follows the **ISO**  *International Organization for Standardization* format. This is the standardization format of date and time [ **YYYY-MM-DD HH:MM:SS** ]

In [146]:
def convert_timestamp(val):
  if pd.isna(val):
    return pd.NaT

  val = str(val).strip() # converts int to string and removes the leading & trailing spaces

  if val.isdigit():   # Numeric Unix timestamp
    return pd.to_datetime(int(val), unit='s')

  return pd.to_datetime(val, errors='coerce')
txn['timestamp'] = txn['timestamp'].apply(convert_timestamp)

### **2.4 amount**
The transaction amount contained inconsistent currency formatting such as currency symbols/text and commas.

In [147]:
# keeps only from [0-9 a-z A-Z] and decimal points and negative symbol
txn['amount'] = (txn['amount'].astype(str).str.replace(r'(?i)rs\.?', '',regex=True)
                 .str.replace(',', '',regex=False).str.strip())

txn['amount'] = pd.to_numeric(txn['amount'], errors='coerce') #(invalid entries safely become NaN)

### **2.5 status**

In [148]:
# need to know what are the status updates
print(txn['status'].dropna().unique())

['COMPLETED' 'TXN_FAILED' 'S' 'TXN_SUCCESS' 'SUCCESS' 'FAILED' 'Success'
 'Initiated' 'Pending' 'Fail' 'Declined' 'F' 'PENDING' 'PROCESSING']


In [149]:
# as we have seen the status has divided into various segments, will be mapped relevant segments
status_map = {
    'COMPLETED': 'Success','TXN_SUCCESS': 'Success','SUCCESS': 'Success','S': 'Success',
    'TXN_FAILED': 'Failed','FAILED': 'Failed','FAIL': 'Failed','DECLINED': 'Failed','F': 'Failed',
    'INITIATED': 'Pending','PENDING': 'Pending','PROCESSING': 'Pending'
}
txn['status'] = txn['status'].map(status_map)
print(txn['status'].value_counts(dropna=True))

status
Success    13625
Failed      1192
Pending      540
Name: count, dtype: int64


In [150]:
print(txn.head())

        txn_id           timestamp   user_id merchant_id    amount  \
0  TXN00011869 2026-01-15 00:11:30  USR45826     MCH7045  15722.34   
1  TXN00010383 2026-01-17 20:09:44  USR79397     MCH5031   6362.90   
2  TXN00008297 2026-01-23 14:10:16  USR87810     MCH9809  15446.19   
3  TXN00006448 2026-02-02 20:17:51  USR54287     MCH6928  12110.49   
4  TXN00018792 2026-03-31 14:02:37  USR53865     MCH8121  19432.94   

             utr     mcc   status  
0  UTR6498104698  5411.0  Success  
1  UTR7656190355  4131.0   Failed  
2  UTR9037001889  5411.0  Success  
3  UTR5257823698  5411.0  Success  
4  UTR4204272894  4131.0  Success  


# **3. KYC File**

In [151]:
k = kyc.copy()
print(k.head())

     user_id        full_name         pan         aadhaar  \
0   USR16112  Dhriti Deshmukh  SEJAA8194O    715658320763   
1   USR17216       Megha Jani  QT0ZZ5561X    023412025028   
2  USR 45454       PANINI LAL  QCNTL9489O  2781 6299 9816   
3   USR46189      Jack Parikh  RRVUI4059Q  0667 8032 7731   
4   USR85256         Eta Ravi  ygvoi9236w    176258817110   

         date_of_birth      city          state monthly_income  occupation  \
0  06/04/1967 12:14 AM    Bombay    Maharashtra          35119     Retired   
1                  NaN   Lucknow  Uttar Pradesh          80907  Freelancer   
2                  NaN   kolkata    West Bengal        ₹11,214      Farmer   
3                  NaN  Amritsar         Punjab          27.3k     Student   
4  1969-02-19 08:25:23     delhi          Delhi     INR 24,705    Salaried   

      signup_timestamp kyc_status risk_segment  
0  2025-12-02 02:18:16       Done          LOW  
1           01-31-2024   Verified       medium  
2  2026-01-30 23:

**As per my view from the data, the columns which required changes are,**

1. **user_id** - inconsistent format [ *need to remove the trailing spaces and proper format STR + integer* ]
2. **full_name** - *Title of the first and second name should be capital*
3. **pan** -  *should be unique, drop the duplicated values because*
4. **aadhar** - *should be unique and remove the trailing spaces between the numbers*
5. **date_of birth & signup_timestamp** - *just maintaining only YYYY-MM-DD fromat because most of the values in column does not have the time format*
6. **monthly_income** - inconsistent symbols [ Rs. , Rs.. , .k] and others
7. State - Title should be capital *italicized text*
8. **kyc_status, risk_segment** - *map the relevant segment categories*

In [152]:
print(k.shape)

(36400, 12)


In [153]:
print(kyc.isna().sum())

user_id                0
full_name              0
pan                 1896
aadhaar             2664
date_of_birth       2944
city                   0
state                  0
monthly_income      2933
occupation             0
signup_timestamp    2910
kyc_status             0
risk_segment           0
dtype: int64


### **3.1 user_id**

In [154]:
k['user_id'] = (k['user_id'].astype('string').str.strip().str.upper())
k['user_id'] = k['user_id'].str.replace(r'\s+', '', regex=True) # remove space btw

# Correct format: USR + 5 digits
k.loc[~k['user_id'].str.fullmatch(r'USR\d{5}', na=False),'user_id'] = pd.NA
k = k.drop_duplicates(subset='user_id', keep='first')


### **3.2 full_name**

In [155]:
k['full_name'] = (k['full_name'].astype(str).str.strip().str.title())

### **3.3 pan**

In [156]:
k['pan'] = (k['pan'].astype('string').str.strip().str.upper())
k = k.drop_duplicates(subset='pan', keep='first')

### **3.4 aadhar**


*   exactly 12 digit integer with unique values
*   no trailing spaces between the integer



In [157]:

k['aadhaar'] = (k['aadhaar'].astype('string').str.replace(r'\D', '', regex=True).str.strip())

# Keep only exactly 12-digit Aadhaar
k.loc[~k['aadhaar'].str.fullmatch(r'\d{12}', na=False),'aadhaar'] = pd.NA
k = k.drop_duplicates(subset='aadhaar', keep='first')

### **3.5 date_of_birth**

In [158]:
k['date_of_birth'] = pd.to_datetime(k['date_of_birth'],errors='coerce')
k['date_of_birth'] = k['date_of_birth'].dt.strftime('%Y-%m-%d')

### **3.6 monthly_income**
*  The values are in inconsistent format, so this code removes [ ₹, INR, Rs., commas, .k, k,
leading and trailing spaces *italicized text* ]
*  whenever we get k , it turns to mathemtical understanding like [ *27.3k to 27300* ]
*  The output will be in the format of string with no other character


In [159]:
k['monthly_income'] = (k['monthly_income'].astype('string').str.strip()
    .str.replace(r'(?i)₹|INR|Rs\.?', '', regex=True).str.replace(',', '', regex=False).str.strip())


k['monthly_income'] = k['monthly_income'].str.replace(
    r'(?i)^(\d+(?:\.\d+)?)k$',
    lambda x: str(float(x.group(1)) * 1000),regex=True)

k['monthly_income'] = pd.to_numeric(k['monthly_income'],errors='coerce')
k['monthly_income'] = k['monthly_income'].fillna(k['monthly_income'].median())

### **3.7 kyc_status** - data mapping

In [160]:
print(k['kyc_status'].dropna().unique())
print(k['kyc_status'].value_counts(dropna=False))


['Done' 'Verified' 'Pending' 'VERIFIED' 'APPROVED' 'REJECTED' 'Reject'
 'Rejected' 'V' 'KYC_DONE' 'Under Review' 'P' 'FAILED' 'IN_PROGRESS' 'R'
 'PENDING']
kyc_status
VERIFIED        2329
V               2320
KYC_DONE        2320
APPROVED        2318
Verified        2242
Done            2216
PENDING          598
Pending          540
P                516
IN_PROGRESS      515
Under Review     514
Rejected         347
Reject           301
REJECTED         290
R                288
FAILED           264
Name: count, dtype: int64


In [161]:
k['kyc_status'] = (k['kyc_status'].astype('string').str.strip().str.upper())

k['kyc_status'] = k['kyc_status'].replace({
    'DONE': 'Verified','VERIFIED': 'Verified','APPROVED': 'Verified','V': 'Verified','KYC_DONE': 'Verified',
    'PENDING': 'Pending','P': 'Pending','IN_PROGRESS': 'Pending','UNDER REVIEW': 'Pending',
    'REJECTED': 'Rejected','REJECT': 'Rejected','R': 'Rejected','FAILED': 'Rejected'})
print(k['kyc_status'].value_counts(dropna=False))

kyc_status
Verified    13745
Pending      2683
Rejected     1490
Name: count, dtype: Int64


### **3.8 risk_segment**

In [162]:
print(k['risk_segment'].dropna().unique())
print(k['risk_segment'].value_counts(dropna=False))

['LOW' 'medium' 'High' 'low' 'MEDIUM' 'high' 'Low' 'Medium' 'Unknown'
 'HIGH' 'UNKNOWN' 'unknown']
risk_segment
low        3502
Low        3459
LOW        3431
medium     1614
Medium     1599
MEDIUM     1560
High        636
high        615
HIGH        607
UNKNOWN     321
Unknown     287
unknown     287
Name: count, dtype: int64


In [163]:
k['risk_segment'] = (k['risk_segment'].astype('string').str.strip().str.upper()
    .replace({
        'low': 'Low','Low': 'Low','LOW': 'Low',
        'MEDIUM': 'Medium','Medium': 'Medium','MEDIUM': 'Medium',
        'HIGH': 'High','High': 'High', 'high': 'High',
        'UNKNOWN': 'High','unknown': 'High', 'Unknown': 'High'
    }))
print(k['risk_segment'].value_counts())

risk_segment
Low       10392
Medium     4773
High       2753
Name: count, dtype: Int64


### **3.9 signup_timstamp**






In [164]:
k['signup_timestamp'] = pd.to_datetime(k['signup_timestamp'],
    errors='coerce',format='mixed').dt.strftime('%Y-%m-%d')

### **3.10 status**

In [165]:
print(k.head())

    user_id        full_name         pan       aadhaar date_of_birth  \
0  USR16112  Dhriti Deshmukh  SEJAA8194O  715658320763    1967-06-04   
1  USR17216       Megha Jani  QT0ZZ5561X  023412025028           NaN   
2  USR45454       Panini Lal  QCNTL9489O  278162999816           NaN   
3  USR46189      Jack Parikh  RRVUI4059Q  066780327731           NaN   
4  USR85256         Eta Ravi  YGVOI9236W  176258817110    1969-02-19   

       city          state  monthly_income  occupation signup_timestamp  \
0    Bombay    Maharashtra         35119.0     Retired       2025-12-02   
1   Lucknow  Uttar Pradesh         80907.0  Freelancer       2024-01-31   
2   kolkata    West Bengal         11214.0      Farmer       2026-01-30   
3  Amritsar         Punjab         27300.0     Student       2025-09-04   
4     delhi          Delhi         24705.0    Salaried       2026-02-03   

  kyc_status risk_segment  
0   Verified          Low  
1   Verified       Medium  
2    Pending         High  
3   

In [166]:
print(k.isna().sum())

user_id                0
full_name              0
pan                    0
aadhaar                1
date_of_birth       2863
city                   0
state                  0
monthly_income         0
occupation             0
signup_timestamp    2838
kyc_status             0
risk_segment           0
dtype: int64



**As from the data insights**
*   2863 person has the same date_of birth - it does not effect the data
*   1 - uniqueness in aadhar, does not effect the respected values
*   signup_timestamp does not effect the values , so we drop the column


In [167]:
k = k.drop(columns=['signup_timestamp'])
print(k.columns)

Index(['user_id', 'full_name', 'pan', 'aadhaar', 'date_of_birth', 'city',
       'state', 'monthly_income', 'occupation', 'kyc_status', 'risk_segment'],
      dtype='object')


# **4. Merchant master File**

In [168]:
mm = merchants.copy()
print(mm.head())

  merchant_id               merchant_name       mcc merchant_category  \
0     mch2849  BHAVSAR, KOTA AND ZACHARIA  MCC-7011     hotel_lodging   
1     MCH4314                Deshmukh Ltd      5699           Apparel   
2     MCH1986     Bains, Chanda and Gh0sh      5311            Retail   
3     mch3899              Goswami-Bath        4131    Transportation   
4     MCH4859                   VASA-RAJU      5812        Restaurant   

     business_type       city      state onboarding_date settlement_account  \
0  Private Limited   Ludhiana     Punjab      09-23-2025                NaN   
1  PRIVATE_LIMITED  Jalandhar     Punjab             NaN         3021439858   
2       individual  Jalandhar     Punjab      10/01/2026                NaN   
3  Sole Proprietor  Hyderabad  Telangana     14-Jan-2023                NaN   
4  SOLE_PROPRIETOR        Hyd  Telangana      1742180385           XXXX9523   

  merchant_status declared_avg_ticket_size  
0        Inactive             INR 2,432.1

In [169]:
print(mm.shape)

(6210, 11)


**As per my view from the data, the columns which required changes are,**
1. **merchant_id** - *inconsistent fromat like [ mch, MCH, Mch, MCh ] - all need to be fromatted to MCH + integer*
2. **merchant name** - *[ remove leading and trailing spaces, title of first and second name should be capital ]*
3. **mcc** - *only less amount of data has the MCH fromat , so we remove it just going to keep the integer*
4. **business_type, merchant_status, City** - *[ data mapping ] - need to be classified to relevant segement*
5. **onboarding_date** - *[ Epoch time and month in str to consistent format - YYYY:MM-DD ]*
6. **declared_avg_ticket_size** - *[ removed the symbols, comma;s , negative and zero value to avg of the column ]*

In [170]:
print(merchants.isna().sum())

merchant_id                    0
merchant_name                  0
mcc                          514
merchant_category              0
business_type                  0
city                           0
state                          0
onboarding_date              499
settlement_account          2451
merchant_status                0
declared_avg_ticket_size     371
dtype: int64


### **4,1 merchant_id**

In [171]:
mm['merchant_id'] = (
    mm['merchant_id']
    .astype('string')
    .str.strip()
    .str.upper()
)
mm = mm.drop_duplicates(subset='merchant_id', keep='first')
print("Removed Duplicate Merchant IDs", mm.shape)

Removed Duplicate Merchant IDs (4872, 11)


In [172]:
mm['merchant_name'] = (mm['merchant_name'].astype('string').str.strip().str.title())

### **4.2 mcc**
Here mcc stands for **Merchant Category Code**

In [173]:
# removed the trailing spaces and inconsistent format
mm['mcc'] = (mm['mcc'].astype('string').str.replace(r'\D', '', regex=True).str.strip())
mm['mcc'] = pd.to_numeric(mm['mcc'], errors='coerce')

### **4.3 business_type**

In [174]:
print(mm['business_type'].dropna().unique())

['Private Limited' 'PRIVATE_LIMITED' 'individual' 'Sole Proprietor'
 'SOLE_PROPRIETOR' 'PARTNERSHIP' 'INDIVIDUAL' 'PRIVATE-LIMITED'
 'partnership' 'SOLE-PROPRIETOR' 'sole_proprietor' 'Partnership'
 'private_limited' 'Individual']


In [175]:
mm['business_type'] = (mm['business_type'].astype('string').str.strip().str.upper()
    .replace({
        'Private Limited': 'Private','private_limited': 'Private',
        'PRIVATE_LIMITED': 'Private','PRIVATE LIMITED': 'Private',
        'PRIVATE-LIMITED': 'Private','Private': 'Private',

        'individual': 'Individual','INDIVIDUAL': 'Individual','Sole Proprieter': 'Sole Proprietor',
        'SOLE_PROPRIETOR': 'Sole Proprietor','sole_proprietor': 'Sole Proprietor',
        'SOLE-PROPRIETOR': 'Sole Proprietor','SOLE PROPRIETOR': 'Sole Proprietor',

        'PARTNERSHIP': 'Partnership','partnership': 'Partnership','Partnership': 'Partnership'
    }))
print(mm['business_type'].value_counts(dropna=False))

business_type
Individual         1250
Sole Proprietor    1226
Private            1207
Partnership        1189
Name: count, dtype: Int64


### **4.4 city**

In [176]:
print(mm['city'].dropna().unique())

['Ludhiana' 'Jalandhar' 'Hyderabad' 'Hyd' 'Chennai' 'JPR' 'chennai'
 'Amritsar' 'jaipur' 'ludhiana' 'Madras' 'lucknow' 'DELHI' 'mumbai'
 'amritsar' 'Jalandar' 'Delhi' 'Mumbai' 'Jaipur' 'kolkata' 'pune' 'Poona'
 'LKO' 'Bangalore' 'Lucknow' 'Calcutta' 'delhi' 'bengaluru' 'LDH' 'BLR'
 'Kolkata' 'Bengaluru' 'Dilli' 'jalandhar' 'Bombay' 'MUMBAI' 'ASR'
 'HYDERABAD' 'New Delhi' 'Pune' 'Mumbay']


In [177]:
mm['city'] = (mm['city'].astype('string').str.strip().str.upper()
    .replace({
        'LUDHIANA': 'Ludhiana','LDH': 'Ludhiana',
        'JALANDHAR': 'Jalandhar','JALANDAR': 'Jalandhar',
        'HYDERABAD': 'Hyderabad','HYD': 'Hyderabad',
        'CHENNAI': 'Chennai','MADRAS': 'Chennai',
        'JAIPUR': 'Jaipur','JPR': 'Jaipur',
        'AMRITSAR': 'Amritsar','ASR': 'Amritsar',
        'LUCKNOW': 'Lucknow','LKO': 'Lucknow',
        'DELHI': 'Delhi','DILLI': 'Delhi','NEW DELHI': 'Delhi',
        'MUMBAI': 'Mumbai','BOMBAY': 'Mumbai','MUMBAY': 'Mumbai',
        'KOLKATA': 'Kolkata','CALCUTTA': 'Kolkata',
        'PUNE': 'Pune','POONA': 'Pune',
        'BANGALORE': 'Bengaluru','BENGALURU': 'Bengaluru','BLR': 'Bengaluru'
    }))
print(mm['city'].value_counts(dropna=False))

city
Chennai      441
Bengaluru    423
Jalandhar    418
Delhi        415
Kolkata      406
Jaipur       401
Pune         400
Hyderabad    399
Lucknow      398
Mumbai       396
Amritsar     394
Ludhiana     381
Name: count, dtype: Int64


### **4.5 onboarding_date**


*   normalize the months [Jan, Feb, Mar] to  generalize fromat
*   remove trailing and leading spaces


In [178]:
def clean_onboarding_date(value):
    if pd.isna(value):
        return pd.NaT
    value = str(value).strip()

    if value.isdigit() and len(value) == 10:
        return pd.to_datetime(int(value), unit='s', errors='coerce')
    return pd.to_datetime(value, errors='coerce')

mm['onboarding_date'] = mm['onboarding_date'].apply(clean_onboarding_date)
mm['onboarding_date'] = mm['onboarding_date'].dt.strftime('%Y-%m-%d')


### **4.6 settlement_account**

In [179]:
mm['settlement_account'] = (mm['settlement_account'].astype('string').str.strip().str.upper())
# Convert blank strings to missing values
mm['settlement_account'] = mm['settlement_account'].replace(r'^\s*$',pd.NA,regex=True)


### **4.7 merchant_status**

In [180]:
print(mm['merchant_status'].dropna().unique())

['Inactive' 'I' 'A' 'ACTIVE' 'Active' 'Live' 'Enabled' 'SUSPENDED' 'Hold'
 'Disabled' 'INACTIVE' 'Closed' 'Blocked' 'Suspended' 'S']


In [181]:
mm['merchant_status'] = (mm['merchant_status'].astype('string').str.strip().str.upper()
    .replace({
        'ACTIVE': 'Active','A': 'Active','LIVE': 'Active','ENABLED': 'Active',
        'INACTIVE': 'Inactive','I': 'Inactive','CLOSED': 'Inactive','DISABLED': 'Inactive',
        'SUSPENDED': 'Suspended','S': 'Suspended','HOLD': 'Suspended','BLOCKED': 'Suspended'
    }))

print(mm['merchant_status'].value_counts(dropna=False))

merchant_status
Active       3997
Inactive      567
Suspended     308
Name: count, dtype: Int64


### **4.8 declared_avg_ticket_size**

* as from the data, some of the data has [ **negative values, zero and positive** ] will be averaged to positive number using **median**
* will handle all the null values
* it will handle the duplicated values from the data

In [182]:
mm['declared_avg_ticket_size'] = (mm['declared_avg_ticket_size'].astype('string').str.strip()
    .str.replace(r'(?i)₹|INR|Rs\.?', '', regex=True)
    .str.replace(',', '', regex=False).str.strip())

mm['declared_avg_ticket_size'] = pd.to_numeric(mm['declared_avg_ticket_size'],errors='coerce')

mm.loc[mm['declared_avg_ticket_size'] < 0,'declared_avg_ticket_size'] = pd.NA
median_ticket = mm['declared_avg_ticket_size'].median()

mm['declared_avg_ticket_size'] = (
    mm['declared_avg_ticket_size'].fillna(median_ticket))

### **4.9 status**

In [183]:
print(mm.head())

  merchant_id               merchant_name     mcc merchant_category  \
0     MCH2849  Bhavsar, Kota And Zacharia  7011.0     hotel_lodging   
1     MCH4314                Deshmukh Ltd  5699.0           Apparel   
2     MCH1986     Bains, Chanda And Gh0Sh  5311.0            Retail   
3     MCH3899                Goswami-Bath  4131.0    Transportation   
4     MCH4859                   Vasa-Raju  5812.0        Restaurant   

     business_type       city      state onboarding_date settlement_account  \
0          Private   Ludhiana     Punjab      2025-09-23               <NA>   
1          Private  Jalandhar     Punjab             NaN         3021439858   
2       Individual  Jalandhar     Punjab      2026-10-01               <NA>   
3  Sole Proprietor  Hyderabad  Telangana      2023-01-14               <NA>   
4  Sole Proprietor  Hyderabad  Telangana      2025-03-17           XXXX9523   

  merchant_status  declared_avg_ticket_size  
0        Inactive                   2432.18  
1     

In [184]:
print(mm.isna().sum())

merchant_id                    0
merchant_name                  0
mcc                          515
merchant_category              0
business_type                  0
city                           0
state                          0
onboarding_date              405
settlement_account          1967
merchant_status                0
declared_avg_ticket_size       0
dtype: int64


# **5. Chargebacks file - json**

In [185]:
js = chargebacks.copy()
print(js.head())

  complaint_id       txn_id   user_id merchant_id transaction_timestamp  \
0   CBK0002082  TXN00004325  usr97580     mch1127            2026/01/28   
1   CBK0001941  TXN00003720  USR54113        3835   25/02/2026 10:24 AM   
2   CBK0001799  TXN00012539  USR17980     mch3700            2026/02/04   
3   CBK0002465  TXN00017802  USR76148     MCH4534           30-Mar-2026   
4   CBK0001870  TXN00015944  USR24660     MCH1686           09-Jan-2026   

  reported_timestamp disputed_amount             reason_code  \
0         02-01-2026                  Merchant Not Delivered   
1         1772691855          414.69       login compromised   
2         02-05-2026       Rs. 7,039          customer issue   
3        01-Apr-2026         1303.05              no service   
4         1768424501         1459.42  merchant service issue   

                                      complaint_text resolution_status  \
0            Customer says amount was debited twice.            CLOSED   
1  User reports 

In [186]:
print(js.shape)

(2884, 13)


### **5.1 user_id**

In [187]:
js['user_id'] = (js['user_id'].astype('string').str.strip().str.upper()
    .str.replace(r'\s+', '', regex=True).str.replace(r'^URR', 'USR', regex=True))


### **5.2 merchant_id**

* in column , some of the value's does not have MCH
*  and also inconsistent format like [ **mch, MCH, McH** ] will deals it to **MCH + integer format**

In [188]:
js['merchant_id'] = (
    js['merchant_id'].astype('string').str.strip()
    .str.upper().str.replace(r'\s+', '', regex=True))

js['merchant_id'] = js['merchant_id'].apply(
    lambda x: 'MCH' + x if pd.notna(x) and not x.startswith('MCH') else x)

### **5.3 complaint_id and txn_id**
*  *here txn_id and complaint_id cannot be duplicate - take example as rms as it deals with unique key fro all the queries*

In [189]:
print("Duplicate complaint_id:",chargebacks['complaint_id'].duplicated().sum())
print("Duplicate txn_id:",chargebacks['txn_id'].duplicated().sum())

Duplicate complaint_id: 84
Duplicate txn_id: 302


In [190]:
js = js.drop_duplicates(subset='txn_id', keep='first')

print("Duplicate txn_id remaining:", js['txn_id'].duplicated().sum())
print("New shape:", js.shape)

Duplicate txn_id remaining: 0
New shape: (2582, 13)


### **5.4 transaction_timestamp, reported_timestamp & bank_response_timestamp**

In [191]:
def clean_date(value):
    if pd.isna(value):
        return pd.NaT
    value = str(value).strip()

    if value.isdigit() and len(value) == 10:
        return pd.to_datetime(int(value), unit='s', errors='coerce')
    return pd.to_datetime(value, errors='coerce')

timestamp_columns = ['transaction_timestamp','reported_timestamp','bank_response_timestamp']

for col in timestamp_columns:
    js[col] = (js[col].apply(clean_date).dt.strftime('%Y-%m-%d'))

### **5.5 disputed_amount**

In [192]:
js['disputed_amount'] = (js['disputed_amount'].astype('string').str.strip()
    .str.replace(r'(?i)₹|INR|Rs\.?', '', regex=True)
    .str.replace(',', '', regex=False).str.strip())

js['disputed_amount'] = pd.to_numeric(js['disputed_amount'],errors='coerce')


### **5.5 complaint_text & reason_code**

In [193]:
# for cleaning complaint_text
js['complaint_text'] = (js['complaint_text'].astype('string').str.replace(r'\bNA\b', '', regex=True)
    .str.replace(r'[^A-Za-z0-9\s]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True).str.strip().str.title())


# for cleaning reason_code
js['reason_code'] = (js['reason_code'].astype('string').str.replace(r'\bNA\b', '', regex=True)
    .str.replace(r'[^A-Za-z0-9\s]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True).str.strip().str.title())

### **5.6 resolution_status**

In [194]:
print(js['resolution_status'].dropna().unique())
print(js['resolution_status'].value_counts(dropna=False))


['CLOSED' 'In Progress' 'OPEN' 'Rejected' 'Closed' 'Open' 'IN_PROGRESS'
 'Resolved' 'WIP' 'RESOLVED' 'REJECTED' 'Pending Bank' 'PENDING_BANK']
resolution_status
Pending Bank    224
CLOSED          222
Rejected        214
Open            208
RESOLVED        206
PENDING_BANK    206
IN_PROGRESS     195
OPEN            193
Closed          191
REJECTED        190
WIP             183
Resolved        181
In Progress     169
Name: count, dtype: int64


In [195]:
js['resolution_status'] = (js['resolution_status'].astype('string').str.strip().str.upper()
    .replace({
        'RESOLVED': 'Resolved','Resolved': 'Resolved','Closed': 'Resolved',

        'Pending Bank': 'Pending','PENDING_BANK': 'Pending','PENDING BANK': 'Pending',
        'OPEN': 'Pending', 'PENDING': 'Pending','IN_PROGRESS': 'Pending','In Progress': 'Pending',
        'Open': 'Pending','WIP': 'Pending',

        'Rejected': 'Rejected','REJECT': 'Rejected','REJECTED': 'Rejected','DECLINED': 'Rejected'
    }))
print(js['resolution_status'].value_counts())

resolution_status
Pending        1209
CLOSED          413
Rejected        404
Resolved        387
IN PROGRESS     169
Name: count, dtype: Int64


### **5.7 severity**

In [196]:
print(js['severity'].dropna().unique())
print(js['severity'].value_counts(dropna=False))

['Critical' 'H' 'P4' 'Low' 'LOW' 'M' 'P2' 'Medium' 'P3' 'CRITICAL'
 'MEDIUM' 'P1' 'HIGH' 'High' 'L' 'CRIT']
severity
P3          246
MEDIUM      237
M           235
Medium      226
P4          225
LOW         221
L           220
Low         214
P2          150
H           138
HIGH        138
High        134
CRIT         59
P1           58
Critical     45
CRITICAL     36
Name: count, dtype: int64


In [197]:
js['severity'] = (js['severity'].astype('string').str.strip()
    .replace({
        'Critical': 'High','CRITICAL': 'High','CRIT': 'High','P1': 'High','P2': 'High','H': 'High','HIGH': 'High',
        'Medium': 'Medium','MEDIUM': 'Medium','M': 'Medium','P3': 'Medium',
        'Low': 'Low','LOW': 'Low','L': 'Low','P4': 'Low'
    }))
print(js['severity'].value_counts())

severity
Medium    944
Low       880
High      758
Name: count, dtype: Int64


### **5.8 channel**

In [198]:
print(js['channel'].dropna().unique())
print(js['channel'].value_counts(dropna=False))

['ivr' 'Call Center' 'IVR' 'App' 'Branch' 'chatbot' 'CHATBOT' 'Email']
channel
Email          350
Branch         339
ivr            331
CHATBOT        320
IVR            318
App            317
chatbot        316
Call Center    291
Name: count, dtype: int64


In [199]:
js['channel'] = (
    js['channel']
    .astype('string')
    .str.strip()
    .replace({
        'ivr': 'Call','IVR': 'Call','Call Center': 'Call',
        'App': 'App',
        'Branch': 'Branch',
        'chatbot': 'Chatbot','CHATBOT': 'Chatbot',
        'Email': 'Email'}))
print(js['channel'].value_counts())

channel
Call       940
Chatbot    636
Email      350
Branch     339
App        317
Name: count, dtype: Int64


### **5.9 status**

In [200]:
print(js.head())

  complaint_id       txn_id   user_id merchant_id transaction_timestamp  \
0   CBK0002082  TXN00004325  USR97580     MCH1127            2026-01-28   
1   CBK0001941  TXN00003720  USR54113     MCH3835            2026-02-25   
2   CBK0001799  TXN00012539  USR17980     MCH3700            2026-02-04   
3   CBK0002465  TXN00017802  USR76148     MCH4534            2026-03-30   
4   CBK0001870  TXN00015944  USR24660     MCH1686            2026-01-09   

  reported_timestamp  disputed_amount             reason_code  \
0         2026-02-01             <NA>  Merchant Not Delivered   
1         2026-03-05           414.69       Login Compromised   
2         2026-02-05           7039.0          Customer Issue   
3         2026-04-01          1303.05              No Service   
4         2026-01-14          1459.42  Merchant Service Issue   

                                      complaint_text resolution_status  \
0             Customer Says Amount Was Debited Twice            CLOSED   
1  User Re

In [201]:
print(js.shape)

(2582, 13)
